# Calibrated absolute scoring vs. direct one-shot scoring

This notebook compares repeated direct 1–5 LLM judgments with the calibrated absolute scorer on the same answers and criteria. It measures **repeatability**, **method agreement**, latency, and model usage.

> Agreement is not accuracy. Without independently created human labels, this experiment cannot establish which method is more correct.

In [5]:
# Copyright (c) 2025 Microsoft Corporation.
from __future__ import annotations

import asyncio
import json
import time
from pathlib import Path

import nest_asyncio
import pandas as pd
from graphrag_common.config import load_config
from IPython.display import display as show

from benchmark_qed.autoe.calibrated import (
    answer_items_from_frame,
    compare_direct_and_calibrated,
    score_direct_baseline,
)
from benchmark_qed.autoe.config import CalibratedAbsoluteScoringConfig
from benchmark_qed.config.model.score import Criteria
from benchmark_qed.llm.factory import ModelFactory

nest_asyncio.apply()

## 1. Configure the experiment

Set `EXPERIMENT_DIR` to the local absolute-scoring project created by the tutorial. Existing calibrated results are reused so the direct baseline can be evaluated without rerunning the calibrated scorer.

In [6]:
EXPERIMENT_DIR = Path(
    "/Users/gaudy-microsoft/Desktop/benchmark-qed_tests/local_new_tests/"
    "calibrated_absolute_test/absolute_scoring"
)
SETTINGS_PATH = EXPERIMENT_DIR / "settings.yaml"
CALIBRATED_RESULTS_PATH = (
    EXPERIMENT_DIR / "output/calibrated_absolute_scores-graphrag_global.csv"
)
DIRECT_RESULTS_PATH = EXPERIMENT_DIR / "output/direct_absolute_trials.csv"
COMPARISON_ITEMS_PATH = EXPERIMENT_DIR / "output/direct_vs_calibrated_items.csv"
COMPARISON_SUMMARY_PATH = EXPERIMENT_DIR / "output/direct_vs_calibrated_summary.csv"
DIRECT_USAGE_PATH = EXPERIMENT_DIR / "output/direct_absolute_model_usage.json"

DIRECT_TRIALS = 3
RUN_DIRECT_BASELINE = True
QUESTION_ID_KEY = "question_id"
QUESTION_TEXT_KEY = "question_text"
ANSWER_TEXT_KEY = "answer"

## 2. Evaluation protocol

- **Direct baseline:** the same configured judge receives one question, one answer, the criterion, and a fixed generic 1–5 rubric. It sees no calibration answers or exemplars. Three calls are made for each answer/criterion.
- **Calibrated scorer:** the judge compares the answer with the frozen five-level scale using the p50, p75, and p25 exemplar sets, then aggregates those three passes.
- Both methods use the same model configuration and the same target answers.
- Calibration construction is a one-time cost and is excluded from frozen-scale scoring. Report it separately when estimating end-to-end deployment cost.

The direct repeats measure ordinary repeated-call stability. The calibrated passes vary the exemplar anchors, so their agreement measures anchor sensitivity. These are related but not identical reliability tests.

In [7]:
config = load_config(CalibratedAbsoluteScoringConfig, SETTINGS_PATH)
if config.input_storage is not None or config.output_storage is not None:
    msg = "This notebook currently supports local input and output storage only."
    raise ValueError(msg)

calibration_path = Path(config.calibration_path)
if not calibration_path.is_absolute():
    calibration_path = EXPERIMENT_DIR / calibration_path
calibration = json.loads(calibration_path.read_text(encoding="utf-8"))
criteria = [
    Criteria.model_validate(scale["criterion"])
    for scale in calibration["scales"].values()
]

targets = []
for condition in config.generated:
    answer_path = Path(condition.answer_base_path)
    if not answer_path.is_absolute():
        answer_path = EXPERIMENT_DIR / answer_path
    answer_frame = pd.read_json(answer_path)
    targets.extend(
        answer_items_from_frame(
            answer_frame,
            condition=condition.name,
            question_id_key=QUESTION_ID_KEY,
            question_text_key=QUESTION_TEXT_KEY,
            answer_text_key=ANSWER_TEXT_KEY,
        )
    )

calibrated_scores = pd.read_csv(CALIBRATED_RESULTS_PATH)
print(
    f"Loaded {len(targets)} targets, {len(criteria)} criteria, and "
    f"{len(calibrated_scores)} calibrated result rows."
)
print(f"Model: {config.llm_config.model}")
print(f"Provider: {config.llm_config.llm_provider}")
print(f"Direct judgments: {len(targets) * len(criteria) * DIRECT_TRIALS}")

Loaded 50 targets, 4 criteria, and 200 calibrated result rows.
Model: gpt-4.1
Provider: azure.openai.chat
Direct judgments: 600


## 3. Run the direct baseline

> **This section makes paid model calls.** With 50 answers, four criteria, and three trials, it makes 600 direct judgments. Review the model configuration and expected cost first, then set `RUN_DIRECT_BASELINE = True`.

In [8]:
if RUN_DIRECT_BASELINE:
    llm_client = ModelFactory.create_chat_model(config.llm_config)
    total_per_criterion = len(targets) * DIRECT_TRIALS
    started = time.perf_counter()
    completed_by_criterion = dict.fromkeys(
        (criterion.name for criterion in criteria), 0
    )
    progress_interval = max(1, total_per_criterion // 10)

    def on_complete(criterion_name: str) -> None:
        """Print progress without requiring notebook widget rendering."""
        completed = completed_by_criterion[criterion_name] + 1
        completed_by_criterion[criterion_name] = completed
        if completed % progress_interval == 0 or completed == total_per_criterion:
            percent = 100 * completed / total_per_criterion
            print(
                f"{criterion_name}: {completed}/{total_per_criterion} ({percent:.0f}%)"
            )

    direct_records = asyncio.get_event_loop().run_until_complete(
        score_direct_baseline(
            llm=llm_client,
            llm_config=config.llm_config,
            targets=targets,
            criteria=criteria,
            trials=DIRECT_TRIALS,
            complete_callback=on_complete,
        )
    )
    direct_wall_seconds = time.perf_counter() - started
    DIRECT_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(direct_records).to_csv(DIRECT_RESULTS_PATH, index=False)
    DIRECT_USAGE_PATH.write_text(
        json.dumps(llm_client.metrics_store.get_metrics(), indent=2),
        encoding="utf-8",
    )
    print(
        f"Saved {len(direct_records)} direct trials in "
        f"{direct_wall_seconds:.1f} seconds to {DIRECT_RESULTS_PATH}."
    )
else:
    print("No model calls made. Set RUN_DIRECT_BASELINE = True to run the baseline.")

relevance: 15/150 (10%)
diversity: 15/150 (10%)
comprehensiveness: 15/150 (10%)
relevance: 30/150 (20%)
empowerment: 15/150 (10%)
diversity: 30/150 (20%)
comprehensiveness: 30/150 (20%)
relevance: 45/150 (30%)
empowerment: 30/150 (20%)
diversity: 45/150 (30%)
comprehensiveness: 45/150 (30%)
relevance: 60/150 (40%)
empowerment: 45/150 (30%)
diversity: 60/150 (40%)
comprehensiveness: 60/150 (40%)
relevance: 75/150 (50%)
empowerment: 60/150 (40%)
diversity: 75/150 (50%)
relevance: 90/150 (60%)
empowerment: 75/150 (50%)
comprehensiveness: 75/150 (50%)
diversity: 90/150 (60%)
empowerment: 90/150 (60%)
comprehensiveness: 90/150 (60%)
relevance: 105/150 (70%)
diversity: 105/150 (70%)
relevance: 120/150 (80%)
comprehensiveness: 105/150 (70%)
empowerment: 105/150 (70%)
diversity: 120/150 (80%)
empowerment: 120/150 (80%)
comprehensiveness: 120/150 (80%)
relevance: 135/150 (90%)
empowerment: 135/150 (90%)
diversity: 135/150 (90%)
comprehensiveness: 135/150 (90%)
relevance: 150/150 (100%)
comprehe

## 4. Compare reliability and method agreement

This cell can be rerun without model calls after the direct trial CSV has been created.

In [9]:
if not DIRECT_RESULTS_PATH.exists():
    msg = f"Run the direct baseline first; no results found at {DIRECT_RESULTS_PATH}."
    raise FileNotFoundError(msg)

direct_trials = pd.read_csv(DIRECT_RESULTS_PATH)
comparison_items, comparison_summary = compare_direct_and_calibrated(
    direct_trials, calibrated_scores
)
comparison_items.to_csv(COMPARISON_ITEMS_PATH, index=False)
comparison_summary.to_csv(COMPARISON_SUMMARY_PATH, index=False)
show(comparison_summary)

,criteria,items,direct_trials,direct_unanimous_rate,direct_pairwise_agreement,direct_mean_latency_seconds,calibrated_vote_agreement,method_exact_agreement,method_within_one,mean_absolute_level_difference
0,comprehensiveness,50,3,0.86,0.906667,193.743553,0.733333,0.18,0.68,1.085106
1,diversity,50,3,0.88,0.920000,180.533082,0.746667,0.16,0.58,1.234043
2,empowerment,50,3,0.82,0.880000,188.141946,0.733333,0.32,0.92,0.680851
3,relevance,50,3,0.90,0.926667,166.588679,0.713333,0.48,0.80,0.760000


In [10]:
print("Direct score distribution")
show(
    pd.pivot_table(
        direct_trials,
        index="criteria",
        columns="level",
        aggfunc="size",
        fill_value=0,
    )
)

print("Direct repeated-call disagreements")
show(
    comparison_items.loc[
        ~comparison_items["direct_unanimous"],
        ["target_id", "criteria", "direct_levels", "direct_level_range"],
    ].sort_values(["direct_level_range", "criteria"], ascending=[False, True])
)

print("Largest direct-vs-calibrated differences")
show(
    comparison_items[
        [
            "target_id",
            "criteria",
            "direct_levels",
            "direct_level",
            "calibrated_level",
            "absolute_level_difference",
        ]
    ].sort_values(  # type: ignore[call-overload] - valid DataFrame sort
        by="absolute_level_difference", ascending=False
    )
)

Direct score distribution


level,1,2,3,4,5
criteria,,,,,
comprehensiveness,9,0,0,61,80
diversity,12,9,21,57,51
empowerment,9,0,0,112,29
relevance,4,0,3,16,127


Direct repeated-call disagreements


,target_id,criteria,direct_levels,direct_level_range
31,graphrag_global:32886ad8-bb42-4605-b9ec-9cd599...,relevance,"[1, 3, 4]",3
24,graphrag_global:2c55c139-5b08-4c67-bb29-75e38f...,comprehensiveness,"[5, 4, 5]",1
32,graphrag_global:34aad71c-7096-484d-a41c-ae43fc...,comprehensiveness,"[5, 4, 4]",1
52,graphrag_global:42c2a269-a8f9-4e8a-bddc-0fe7f8...,comprehensiveness,"[4, 5, 5]",1
60,graphrag_global:4421efb2-611b-48b8-b122-08fc82...,comprehensiveness,"[5, 4, 4]",1
72,graphrag_global:610b37b6-3464-44cf-a827-d64d2f...,comprehensiveness,"[4, 4, 5]",1
96,graphrag_global:8dc4ab52-a24c-41b5-b491-d2b65e...,comprehensiveness,"[5, 4, 5]",1
168,graphrag_global:e3ea64fc-1e7c-49ba-9df8-b26dd5...,comprehensiveness,"[5, 4, 5]",1
33,graphrag_global:34aad71c-7096-484d-a41c-ae43fc...,diversity,"[4, 4, 5]",1
49,graphrag_global:3f92cc14-9904-4833-84cb-2afc0e...,diversity,"[4, 3, 3]",1


Largest direct-vs-calibrated differences


,target_id,criteria,direct_levels,direct_level,calibrated_level,absolute_level_difference
11,graphrag_global:1c1789a6-2dc8-42ec-b089-55ca75...,relevance,"[5, 5, 5]",5,1.0,4.0
13,graphrag_global:20fbde7b-7613-462d-9120-92472b...,diversity,"[5, 5, 5]",5,2.0,3.0
104,graphrag_global:95b4f12a-6a49-4d60-bcaf-457ee6...,comprehensiveness,"[4, 4, 4]",4,2.0,2.0
129,graphrag_global:b6be55fd-3985-4d94-8589-db6d16...,diversity,"[4, 4, 4]",4,2.0,2.0
117,graphrag_global:a4d2c69a-7d4f-4248-b261-2f141f...,diversity,"[5, 5, 5]",5,3.0,2.0
...,...,...,...,...,...,...
29,graphrag_global:32886ad8-bb42-4605-b9ec-9cd599...,diversity,"[1, 1, 1]",1,NaN,NaN
30,graphrag_global:32886ad8-bb42-4605-b9ec-9cd599...,empowerment,"[1, 1, 1]",1,NaN,NaN
56,graphrag_global:44136e73-2bc7-4975-b337-f5c1e3...,comprehensiveness,"[1, 1, 1]",1,NaN,NaN
57,graphrag_global:44136e73-2bc7-4975-b337-f5c1e3...,diversity,"[1, 1, 1]",1,NaN,NaN


## 5. Interpret the results

- `direct_unanimous_rate`: fraction of answer/criterion cases where all direct calls assigned the same level.
- `direct_pairwise_agreement`: average fraction of direct-call pairs that assigned the same level.
- `calibrated_vote_agreement`: agreement among the p50/p75/p25 exemplar passes.
- `method_exact_agreement`: fraction where the direct consensus equals the calibrated level.
- `method_within_one`: fraction where the methods differ by at most one level.
- `mean_absolute_level_difference`: average ordinal distance between methods.

Per-request direct latency is recorded, but wall-clock latency is affected by provider throttling and concurrency. Compare token/request metrics from `direct_absolute_model_usage.json` and the calibrated scorer's `model_usage.json`; include calibration construction separately as a one-time cost.

**Do not interpret cross-method agreement as accuracy.** To evaluate correctness, create a stratified human-gold set with examples near every level boundary, blind the annotators to method outputs, measure inter-annotator agreement, and compare both methods against the adjudicated labels.